In [1]:
# ===================================================
# WEEK 3 DAY 2 - CI/CD Pipeline 
# Phetho Tlaka | May 2026
# 
# CI = Continuos Intergration
#      Every push to GitHub automatically
#      - Runs tests
#      - Build the Docker image
#      - Checks for errors
#
# CD = Continuous Deployment
#      If CI passes automatically:
#      - Pushes image to registry 
#      - Deploy to Kubernetes
#      - Your app is live with new code
#
# TOOL: GitHub Actions
#       Free built into GitHub
#       Used by millions of companies worldwide
#       Triger by git push
# ==========================================================
print("=" * 52)
print("  WEEK 3 DAY 2 -CI/CD Pipeline")
print("=" * 52)
print()
print("The CI/CD Flow:")
print()
print("  You write code")
print("  → git add . && git commit && git push")
print("  → GitHub Actions triggers automatically")
print("  → Step 1: Run tests (CI)")
print("  → Step 2: Build Docker image (CI)")
print("  → Step 3: Push to Docker Hub (CD)")
print("  → Step 4: Deploy to Kubernetes (CD)")
print("  → Your live app is updated")
print("  → Total time: ~3 minutes")
print("  → Human steps: ZERO")
print()
print("Tools:")
print("  GitHub Actions  — runs the pipeline")
print("  Docker Hub      — stores your images")
print("  kubectl         — deploys to K8s")
print()
print("Today we build this pipeline")
print("for the titanic-api.")
      

  WEEK 3 DAY 2 -CI/CD Pipeline

The CI/CD Flow:

  You write code
  → git add . && git commit && git push
  → GitHub Actions triggers automatically
  → Step 1: Run tests (CI)
  → Step 2: Build Docker image (CI)
  → Step 3: Push to Docker Hub (CD)
  → Step 4: Deploy to Kubernetes (CD)
  → Your live app is updated
  → Total time: ~3 minutes
  → Human steps: ZERO

Tools:
  GitHub Actions  — runs the pipeline
  Docker Hub      — stores your images
  kubectl         — deploys to K8s

Today we build this pipeline
for your titanic-api.


In [2]:
# ================================================
# STEP 1: I NEED TO WRITE TESTS FIRST
# ================================================
# Before I build a CI/CD pipeline, I need tests.
# CI/CD without tests is pointless — the pipeline
# would deploy broken code automatically.
#
# I will write tests for my FastAPI endpoints using
# pytest — the standard Python testing framework.
#
# What I am testing:
# 1. My health check endpoint returns 200
# 2. My predict endpoint returns a valid prediction
# 3. My predict endpoint rejects invalid input
#
# A test passes when the result matches what I expect.
# A test fails when something is broken.
# If any test fails → my pipeline STOPS → no deploy.
# This protects my live app from broken code.

import os
import sys

print("=" * 52)
print("  STEP 1: WRITING TESTS FOR MY API")
print("=" * 52)
print()
print("Why I write tests before CI/CD:")
print()
print("  Scenario WITHOUT tests:")
print("  I push broken code → pipeline deploys it")
print("  → my live API crashes → users see errors")
print()
print("  Scenario WITH tests:")
print("  I push broken code → tests fail")
print("  → pipeline stops → broken code NEVER deploys")
print("  → my live API keeps running perfectly")
print()
print("Test framework: pytest")
print("Test file:      ml/test_api.py")
print()
print("Tests I will write:")
print("  Test 1: GET /  returns status 200")
print("  Test 2: GET /  returns 'healthy' status")
print("  Test 3: POST /predict returns SURVIVED or DIED")
print("  Test 4: POST /predict with invalid pclass → 422")
print("  Test 5: POST /predict with invalid sex → 422")
print()

# Check pytest is available
result = os.popen('pip show pytest 2>/dev/null').read()
if 'pytest' in result:
    print("pytest is already installed ✓")
else:
    print("Installing pytest...")
    os.system('pip install pytest httpx --quiet')
    print("pytest installed ✓")

  STEP 1: WRITING TESTS FOR MY API

Why I write tests before CI/CD:

  Scenario WITHOUT tests:
  I push broken code → pipeline deploys it
  → my live API crashes → users see errors

  Scenario WITH tests:
  I push broken code → tests fail
  → pipeline stops → broken code NEVER deploys
  → my live API keeps running perfectly

Test framework: pytest
Test file:      ml/test_api.py

Tests I will write:
  Test 1: GET /  returns status 200
  Test 2: GET /  returns 'healthy' status
  Test 3: POST /predict returns SURVIVED or DIED
  Test 4: POST /predict with invalid pclass → 422
  Test 5: POST /predict with invalid sex → 422

pytest is already installed ✓


In [3]:
# ================================================
# STEP 2: I AM WRITING MY TEST FILE
# ================================================
# I am saving this as ml/test_api.py
# pytest will automatically find and run any file
# that starts with "test_"
#
# I am using FastAPI's TestClient which lets me
# send HTTP requests to my API without actually
# starting a server — perfect for automated testing
#
# Each test function must start with "test_"
# pytest finds them automatically and runs them all

test_code = '''import pytest
from fastapi.testclient import TestClient
import sys
import os

# I add the ml folder to my path so Python
# can find my main.py file
sys.path.insert(0, os.path.dirname(__file__))

# I import my FastAPI app from main.py
from main import app

# I create a test client — this lets me send
# HTTP requests without starting a real server
client = TestClient(app)


# ── TEST 1 ───────────────────────────────────────
# I am testing that my health check endpoint
# returns HTTP status 200 (OK)
def test_health_check_status():
    response = client.get("/")
    assert response.status_code == 200, (
        f"Expected 200 but got {response.status_code}"
    )
    print("✓ Health check returns 200")


# ── TEST 2 ───────────────────────────────────────
# I am testing that my health check returns
# the word "healthy" in the response body
def test_health_check_content():
    response = client.get("/")
    data = response.json()
    assert data["status"] == "healthy", (
        f"Expected healthy but got {data['status']}"
    )
    print("✓ Health check returns healthy status")


# ── TEST 3 ───────────────────────────────────────
# I am testing a valid prediction request
# A wealthy woman in 1st class should SURVIVE
def test_predict_valid_passenger():
    response = client.post("/predict", json={
        "pclass":   1,
        "sex":      "female",
        "age":      25,
        "sibsp":    0,
        "parch":    0,
        "fare":     100.0,
        "embarked": "C"
    })
    assert response.status_code == 200
    data = response.json()

    # I check all expected fields exist
    assert "survived"   in data
    assert "prediction" in data
    assert "confidence" in data

    # I check prediction is one of the valid values
    assert data["prediction"] in ["SURVIVED", "DIED"], (
        f"Unexpected prediction: {data['prediction']}"
    )

    # I check probability is between 0 and 1
    assert 0 <= data["probability"] <= 1

    print(f"✓ Valid prediction: {data['prediction']}")
    print(f"  Confidence: {data['confidence']}")


# ── TEST 4 ───────────────────────────────────────
# I am testing that invalid pclass (5) is rejected
# My API should return 422 Unprocessable Entity
def test_predict_invalid_pclass():
    response = client.post("/predict", json={
        "pclass":   5,
        "sex":      "female",
        "age":      25,
        "sibsp":    0,
        "parch":    0,
        "fare":     100.0,
        "embarked": "C"
    })
    assert response.status_code == 422, (
        f"Expected 422 but got {response.status_code}"
    )
    print("✓ Invalid pclass=5 correctly rejected (422)")


# ── TEST 5 ───────────────────────────────────────
# I am testing that invalid sex is rejected
# "alien" is not male or female
def test_predict_invalid_sex():
    response = client.post("/predict", json={
        "pclass":   1,
        "sex":      "alien",
        "age":      25,
        "sibsp":    0,
        "parch":    0,
        "fare":     100.0,
        "embarked": "C"
    })
    assert response.status_code == 422, (
        f"Expected 422 but got {response.status_code}"
    )
    print("✓ Invalid sex=alien correctly rejected (422)")


# ── TEST 6 ───────────────────────────────────────
# I am testing a poor man in 3rd class
# He should very likely DIE
def test_predict_poor_man():
    response = client.post("/predict", json={
        "pclass":   3,
        "sex":      "male",
        "age":      30,
        "sibsp":    0,
        "parch":    0,
        "fare":     8.0,
        "embarked": "S"
    })
    assert response.status_code == 200
    data = response.json()
    assert data["prediction"] == "DIED", (
        f"Expected DIED but got {data['prediction']}"
    )
    print(f"✓ Poor 3rd class man correctly predicted DIED")


# ── TEST 7 ───────────────────────────────────────
# I am testing the model-info endpoint
def test_model_info():
    response = client.get("/model-info")
    assert response.status_code == 200
    data = response.json()
    assert "model_type"   in data
    assert "n_estimators" in data
    assert "features"     in data
    assert data["n_estimators"] == 100
    print(f"✓ Model info: {data['model_type']}")
    print(f"  Estimators: {data['n_estimators']}")
'''

# I save the test file to my ml folder
test_path = '/home/phetho/projects/phetho-lab/ml/test_api.py'
with open(test_path, 'w') as f:
    f.write(test_code)

print("Test file saved to: ml/test_api.py")
print()
print("I have written 7 tests:")
print("  test_health_check_status   — GET / returns 200")
print("  test_health_check_content  — GET / returns healthy")
print("  test_predict_valid_passenger — valid prediction works")
print("  test_predict_invalid_pclass  — pclass=5 rejected")
print("  test_predict_invalid_sex     — sex=alien rejected")
print("  test_predict_poor_man        — poor man predicts DIED")
print("  test_model_info              — model-info works")
print()
print("Next: I will run these tests with pytest")

Test file saved to: ml/test_api.py

I have written 7 tests:
  test_health_check_status   — GET / returns 200
  test_health_check_content  — GET / returns healthy
  test_predict_valid_passenger — valid prediction works
  test_predict_invalid_pclass  — pclass=5 rejected
  test_predict_invalid_sex     — sex=alien rejected
  test_predict_poor_man        — poor man predicts DIED
  test_model_info              — model-info works

Next: I will run these tests with pytest


In [4]:
# ================================================
# STEP 3: I AM RUNNING MY TESTS WITH PYTEST
# ================================================
# I run pytest from inside my ml folder
# pytest automatically finds test_api.py because
# the filename starts with "test_"
#
# -v flag means verbose — I see each test name
# and whether it passed or failed
#
# Green = PASSED ✓
# Red   = FAILED ✗
#
# I need ALL 7 tests to pass before I build
# my CI/CD pipeline — if any fail the pipeline
# would catch them automatically on GitHub

import subprocess
import os

print("=" * 52)
print("  RUNNING MY TESTS")
print("=" * 52)
print()
print("Command: pytest ml/test_api.py -v")
print()

result = subprocess.run(
    ['python', '-m', 'pytest',
     'ml/test_api.py',
     '-v',
     '--tb=short'],
    cwd='/home/phetho/projects/phetho-lab',
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode == 0:
    print("=" * 52)
    print("  ALL TESTS PASSED ✓")
    print("  My API is working correctly")
    print("  I am ready to build my CI/CD pipeline")
    print("=" * 52)
else:
    print(result.stderr)
    print("=" * 52)
    print("  SOME TESTS FAILED")
    print("  I need to fix these before CI/CD")
    print("=" * 52)
    

  RUNNING MY TESTS

Command: pytest ml/test_api.py -v

============================= test session starts ==============================
platform linux -- Python 3.12.7, pytest-7.4.4, pluggy-1.0.0 -- /home/phetho/anaconda3/bin/python
cachedir: .pytest_cache
rootdir: /home/phetho/projects/phetho-lab
plugins: anyio-4.2.0
collecting ... collected 0 items / 1 error

==================================== ERRORS ====================================
_______________________ ERROR collecting ml/test_api.py ________________________
ml/test_api.py:11: in <module>
    from main import app
ml/main.py:20: in <module>
    raise FileNotFoundError(
E   FileNotFoundError: Model not found at models/titanic_model.pkl. Run week2_day2_model_api.ipynb first.
=========================== short test summary info ============================
ERROR ml/test_api.py - FileNotFoundError: Model not found at models/titanic_model.pkl. Run week2_d...
!!!!!!!!!!!!!!!!!!!! Interrupted: 1 error during collection !!!!!!!!!!!!!

In [5]:
# ================================================
# STEP 4: I AM FIXING MY TEST FILE
# ================================================
# The problem: when pytest runs from the repo root
# it looks for models/titanic_model.pkl
# but my model is at ml/models/titanic_model.pkl
#
# The fix: I change the working directory to ml/
# before importing my app so it finds the model
# This is a common pattern in Python testing

test_code = '''import pytest
import os
import sys

# I change into the ml directory first
# so main.py can find models/titanic_model.pkl
os.chdir(os.path.dirname(os.path.abspath(__file__)))

# I add ml/ to path so Python finds main.py
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

from fastapi.testclient import TestClient
from main import app

client = TestClient(app)


# ── TEST 1 ───────────────────────────────────────
# I test my health check returns 200
def test_health_check_status():
    response = client.get("/")
    assert response.status_code == 200
    print("✓ Health check returns 200")


# ── TEST 2 ───────────────────────────────────────
# I test my health check returns healthy status
def test_health_check_content():
    response = client.get("/")
    data = response.json()
    assert data["status"] == "healthy"
    print("✓ Health check returns healthy status")


# ── TEST 3 ───────────────────────────────────────
# I test a valid prediction — wealthy woman
# in 1st class should SURVIVE
def test_predict_valid_passenger():
    response = client.post("/predict", json={
        "pclass":   1,
        "sex":      "female",
        "age":      25,
        "sibsp":    0,
        "parch":    0,
        "fare":     100.0,
        "embarked": "C"
    })
    assert response.status_code == 200
    data = response.json()
    assert "survived"   in data
    assert "prediction" in data
    assert "confidence" in data
    assert data["prediction"] in ["SURVIVED", "DIED"]
    assert 0 <= data["probability"] <= 1
    print(f"✓ Valid prediction: {data['prediction']}")


# ── TEST 4 ───────────────────────────────────────
# I test that pclass=5 is rejected with 422
def test_predict_invalid_pclass():
    response = client.post("/predict", json={
        "pclass":   5,
        "sex":      "female",
        "age":      25,
        "sibsp":    0,
        "parch":    0,
        "fare":     100.0,
        "embarked": "C"
    })
    assert response.status_code == 422
    print("✓ Invalid pclass=5 correctly rejected (422)")


# ── TEST 5 ───────────────────────────────────────
# I test that sex=alien is rejected with 422
def test_predict_invalid_sex():
    response = client.post("/predict", json={
        "pclass":   1,
        "sex":      "alien",
        "age":      25,
        "sibsp":    0,
        "parch":    0,
        "fare":     100.0,
        "embarked": "C"
    })
    assert response.status_code == 422
    print("✓ Invalid sex=alien correctly rejected (422)")


# ── TEST 6 ───────────────────────────────────────
# I test a poor man in 3rd class — should DIE
def test_predict_poor_man():
    response = client.post("/predict", json={
        "pclass":   3,
        "sex":      "male",
        "age":      30,
        "sibsp":    0,
        "parch":    0,
        "fare":     8.0,
        "embarked": "S"
    })
    assert response.status_code == 200
    data = response.json()
    assert data["prediction"] == "DIED"
    print("✓ Poor 3rd class man correctly predicted DIED")


# ── TEST 7 ───────────────────────────────────────
# I test my model-info endpoint
def test_model_info():
    response = client.get("/model-info")
    assert response.status_code == 200
    data = response.json()
    assert "model_type"   in data
    assert "n_estimators" in data
    assert "features"     in data
    assert data["n_estimators"] == 100
    print(f"✓ Model info returns correctly")
'''

# I overwrite the old test file with the fixed version
test_path = '/home/phetho/projects/phetho-lab/ml/test_api.py'
with open(test_path, 'w') as f:
    f.write(test_code)

print("Fixed test file saved!")
print()
print("What I fixed:")
print("  Added os.chdir() to move into ml/ folder")
print("  So main.py finds models/titanic_model.pkl")
print("  Added absolute path resolution with")
print("  os.path.abspath(__file__)")
print()
print("Now running tests again...")
print()

import subprocess

result = subprocess.run(
    ['python', '-m', 'pytest',
     'ml/test_api.py',
     '-v',
     '--tb=short'],
    cwd='/home/phetho/projects/phetho-lab',
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode == 0:
    print("=" * 52)
    print("  ALL 7 TESTS PASSED ✓")
    print("  My API is verified and working")
    print("  I am ready to build CI/CD pipeline")
    print("=" * 52)
else:
    print(result.stderr[-1000:])
    print("=" * 52)
    print("  TESTS STILL FAILING")
    print("=" * 52)

Fixed test file saved!

What I fixed:
  Added os.chdir() to move into ml/ folder
  So main.py finds models/titanic_model.pkl
  Added absolute path resolution with
  os.path.abspath(__file__)

Now running tests again...

============================= test session starts ==============================
platform linux -- Python 3.12.7, pytest-7.4.4, pluggy-1.0.0 -- /home/phetho/anaconda3/bin/python
cachedir: .pytest_cache
rootdir: /home/phetho/projects/phetho-lab
plugins: anyio-4.2.0
collecting ... collected 7 items

ml/test_api.py::test_health_check_status PASSED                          [ 14%]
ml/test_api.py::test_health_check_content PASSED                         [ 28%]
ml/test_api.py::test_predict_valid_passenger PASSED                      [ 42%]
ml/test_api.py::test_predict_invalid_pclass PASSED                       [ 57%]
ml/test_api.py::test_predict_invalid_sex PASSED                          [ 71%]
ml/test_api.py::test_predict_poor_man PASSED                             [ 85%]
m

In [6]:
# ================================================
# STEP 5: I AM BUILDING MY GITHUB ACTIONS PIPELINE
# ================================================
# GitHub Actions is a CI/CD tool built into GitHub
# It reads a YAML file I put in .github/workflows/
# and runs it automatically every time I push code
#
# MY PIPELINE WILL DO THIS ON EVERY GIT PUSH:
#
# Job 1 — TEST (CI):
#   - Check out my code
#   - Install Python and dependencies
#   - Run my 7 pytest tests
#   - If any fail — STOP, do not deploy
#
# Job 2 — BUILD (CI):
#   - Only runs if Job 1 passed
#   - Build my Docker image
#   - Push it to Docker Hub
#
# YAML FORMAT:
#   YAML uses indentation (spaces, not tabs)
#   Each level = 2 spaces
#   - means a list item
#   key: value means a setting

import os

pipeline = '''# ================================================
# MY GITHUB ACTIONS CI/CD PIPELINE
# Phetho Tlaka | Week 3 Day 2
#
# This file lives at .github/workflows/cicd.yml
# GitHub reads it automatically on every push
# ================================================

name: Titanic API CI/CD Pipeline

# I trigger this pipeline on every push to main
on:
  push:
    branches: [ main ]
  pull_request:
    branches: [ main ]

jobs:

  # ── JOB 1: TEST ────────────────────────────────
  # I run my pytest tests first
  # If any test fails the build stops here
  test:
    name: Run Tests
    runs-on: ubuntu-latest

    steps:
      # Step 1: I check out my code from GitHub
      - name: Checkout code
        uses: actions/checkout@v4

      # Step 2: I set up Python 3.11
      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      # Step 3: I install my dependencies
      - name: Install dependencies
        run: |
          pip install --upgrade pip
          pip install fastapi uvicorn scikit-learn
          pip install pandas joblib pydantic
          pip install pytest httpx

      # Step 4: I train a fresh model for testing
      # I need the model file to exist before tests run
      - name: Train model for testing
        run: |
          cd ml
          python -c "
          import pandas as pd
          import numpy as np
          from sklearn.ensemble import RandomForestClassifier
          from sklearn.model_selection import train_test_split
          import joblib, os

          df = pd.read_csv('../analytics/titanic.csv')
          df = df.drop(columns=['Cabin'])
          df['Age'] = df['Age'].fillna(df['Age'].median())
          df['Embarked'] = df['Embarked'].fillna('S')
          df['Sex_num'] = df['Sex'].map({'male':0,'female':1})
          df['Embarked_num'] = df['Embarked'].map({'S':0,'C':1,'Q':2})

          features = ['Pclass','Sex_num','Age',
                      'SibSp','Parch','Fare','Embarked_num']
          X = df[features]
          y = df['Survived']

          X_train,X_test,y_train,y_test = (
              __import__('sklearn.model_selection',
              fromlist=['train_test_split'])
              .train_test_split(X,y,test_size=0.2,
              random_state=42))

          model = RandomForestClassifier(
              n_estimators=100, random_state=42)
          model.fit(X_train, y_train)

          os.makedirs('models', exist_ok=True)
          joblib.dump(model, 'models/titanic_model.pkl')
          joblib.dump(features, 'models/features.pkl')
          print('Model trained and saved')
          "

      # Step 5: I run my 7 pytest tests
      - name: Run pytest
        run: |
          python -m pytest ml/test_api.py -v

  # ── JOB 2: BUILD AND PUSH DOCKER IMAGE ─────────
  # I only build if all my tests passed
  # I push my Docker image to Docker Hub
  build:
    name: Build and Push Docker Image
    runs-on: ubuntu-latest
    needs: test
    if: github.ref == \'refs/heads/main\'

    steps:
      # Step 1: I check out my code
      - name: Checkout code
        uses: actions/checkout@v4

      # Step 2: I log into Docker Hub
      # I store my password as a GitHub Secret
      # Secrets are encrypted — never visible in logs
      - name: Login to Docker Hub
        uses: docker/login-action@v3
        with:
          username: ${{ secrets.DOCKERHUB_USERNAME }}
          password: ${{ secrets.DOCKERHUB_TOKEN }}

      # Step 3: I train model before building image
      - name: Set up Python and train model
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Train model for Docker build
        run: |
          pip install scikit-learn pandas joblib numpy
          cd ml
          python -c "
          import pandas as pd
          from sklearn.ensemble import RandomForestClassifier
          from sklearn.model_selection import train_test_split
          import joblib, os

          df = pd.read_csv('../analytics/titanic.csv')
          df = df.drop(columns=['Cabin'])
          df['Age'] = df['Age'].fillna(df['Age'].median())
          df['Embarked'] = df['Embarked'].fillna('S')
          df['Sex_num'] = df['Sex'].map({'male':0,'female':1})
          df['Embarked_num'] = df['Embarked'].map({'S':0,'C':1,'Q':2})
          features = ['Pclass','Sex_num','Age',
                      'SibSp','Parch','Fare','Embarked_num']
          model = RandomForestClassifier(
              n_estimators=100,random_state=42)
          model.fit(df[features], df['Survived'])
          os.makedirs('models', exist_ok=True)
          joblib.dump(model,'models/titanic_model.pkl')
          joblib.dump(features,'models/features.pkl')
          print('Model ready')
          "

      # Step 4: I build my Docker image and push it
      # pmtee/titanic-api:latest is my image name
      # I also tag it with the git commit SHA
      # So I can always roll back to any previous version
      - name: Build and push Docker image
        uses: docker/build-push-action@v5
        with:
          context: ./ml
          push: true
          tags: |
            pmtee/titanic-api:latest
            pmtee/titanic-api:${{ github.sha }}
'''

# I create the .github/workflows directory
workflows_dir = '/home/phetho/projects/phetho-lab/.github/workflows'
os.makedirs(workflows_dir, exist_ok=True)

# I save the pipeline file
pipeline_path = f'{workflows_dir}/cicd.yml'
with open(pipeline_path, 'w') as f:
    f.write(pipeline)

print("Pipeline file created!")
print()
print(f"Location: .github/workflows/cicd.yml")
print(f"Size:     {len(pipeline)} characters")
print()
print("What this pipeline does on every git push:")
print()
print("  Job 1 — TEST (runs first):")
print("    ✓ Checks out my code from GitHub")
print("    ✓ Installs Python 3.11")
print("    ✓ Installs all dependencies")
print("    ✓ Trains a fresh ML model")
print("    ✓ Runs all 7 pytest tests")
print("    ✗ Stops if any test fails")
print()
print("  Job 2 — BUILD (only if Job 1 passed):")
print("    ✓ Logs into Docker Hub")
print("    ✓ Builds my Docker image")
print("    ✓ Pushes pmtee/titanic-api:latest")
print("    ✓ Tags with git commit SHA")
print()
print("Next: I need to set up Docker Hub")
print("and add my secrets to GitHub")

Pipeline file created!

Location: .github/workflows/cicd.yml
Size:     5228 characters

What this pipeline does on every git push:

  Job 1 — TEST (runs first):
    ✓ Checks out my code from GitHub
    ✓ Installs Python 3.11
    ✓ Installs all dependencies
    ✓ Trains a fresh ML model
    ✓ Runs all 7 pytest tests
    ✗ Stops if any test fails

  Job 2 — BUILD (only if Job 1 passed):
    ✓ Logs into Docker Hub
    ✓ Builds my Docker image
    ✓ Pushes pmtee/titanic-api:latest
    ✓ Tags with git commit SHA

Next: I need to set up Docker Hub
and add my secrets to GitHub
